In [19]:
# ── Cell 1 — Libraries ────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


In [20]:
# ── Cell 2 — Load Demand Table ────────────────────────────────────────
demand_df = pd.read_csv(
    r'F:\projectss\personal\blinkit_project\data\processed\demand_table.csv'
)

print(f"Shape   : {demand_df.shape}")
print(f"Columns : {demand_df.columns.tolist()}")
print(demand_df.head(3))

Shape   : (79213, 10)
Columns : ['product_id', 'product_name', 'pincode', 'area_name', 'latitude', 'longitude', 'order_dow', 'order_hour_of_day', 'demand', 'demand_log']
   product_id   product_name  pincode area_name   latitude  longitude  \
0        4605  Yellow Onions   380006     Paldi  23.018508  72.569819   
1        4605  Yellow Onions   380006     Paldi  23.018508  72.569819   
2        4605  Yellow Onions   380006     Paldi  23.018508  72.569819   

   order_dow  order_hour_of_day  demand  demand_log  
0          0                  0     100    4.615121  
1          0                  1     120    4.795791  
2          0                  2      50    3.931826  


In [21]:
# ── Cell 3 — Build Time Series ────────────────────────────────────────
NUM_WEEKS  = 26
START_DATE = '2024-01-01'

all_rows = []

for week in range(NUM_WEEKS):
    week_start   = pd.Timestamp(START_DATE) + pd.Timedelta(weeks=week)
    noise_factor = np.random.uniform(0.60, 1.40)

    for _, row in demand_df.iterrows():
        demand     = round(max(0, row['demand']     * noise_factor))
        demand_log = round(np.log1p(demand), 4)

        dt = week_start + pd.Timedelta(
            days =int(row['order_dow']),
            hours=int(row['order_hour_of_day'])
        )

        all_rows.append({
            'datetime'          : dt,
            'week_number'       : week + 1,
            'product_id'        : row['product_id'],
            'product_name'      : row['product_name'],
            'pincode'           : row['pincode'],
            'area_name'         : row['area_name'],
            'latitude'          : row['latitude'],
            'longitude'         : row['longitude'],
            'order_dow'         : row['order_dow'],
            'order_hour_of_day' : row['order_hour_of_day'],
            'demand'            : float(demand),
            'demand_log'        : float(demand_log),
        })

ts_df = pd.DataFrame(all_rows)
ts_df = ts_df.sort_values(
    ['product_id','pincode','datetime']
).reset_index(drop=True)

print(f"Time series shape : {ts_df.shape}")
print(f"Date range        : {ts_df['datetime'].min()} → {ts_df['datetime'].max()}")
print(ts_df[['datetime','product_id','pincode','demand','demand_log']].head(5))

Time series shape : (2059538, 12)
Date range        : 2024-01-01 00:00:00 → 2024-06-30 23:00:00
             datetime  product_id  pincode  demand  demand_log
0 2024-01-01 00:00:00        4605   380006    78.0      4.3694
1 2024-01-01 01:00:00        4605   380006    93.0      4.5433
2 2024-01-01 02:00:00        4605   380006    39.0      3.6889
3 2024-01-01 04:00:00        4605   380006    39.0      3.6889
4 2024-01-01 05:00:00        4605   380006    31.0      3.4657


In [22]:
# ── Cell 4 — Leakage Check ────────────────────────────────────────────
grp = ts_df.groupby(['product_id','pincode'])['demand']
ts_df['lag_168'] = grp.shift(168)

leakage = (ts_df['lag_168'] == ts_df['demand']).mean()
print(f"Leakage check : {leakage*100:.2f}%")
print("Must be near 0% — if so we are good!")

Leakage check : 0.30%
Must be near 0% — if so we are good!


In [23]:
# ── Cell 5 — Cyclical Time Features ──────────────────────────────────
ts_df['hour_sin'] = np.sin(2 * np.pi * ts_df['order_hour_of_day'] / 24)
ts_df['hour_cos'] = np.cos(2 * np.pi * ts_df['order_hour_of_day'] / 24)
ts_df['dow_sin']  = np.sin(2 * np.pi * ts_df['order_dow'] / 7)
ts_df['dow_cos']  = np.cos(2 * np.pi * ts_df['order_dow'] / 7)
ts_df['week_sin'] = np.sin(2 * np.pi * ts_df['week_number'] / 52)
ts_df['week_cos'] = np.cos(2 * np.pi * ts_df['week_number'] / 52)
ts_df['month']    = ts_df['datetime'].dt.month

print("Cyclical features added!")
print(ts_df[['order_hour_of_day','hour_sin','hour_cos']].head(3))

Cyclical features added!
   order_hour_of_day  hour_sin  hour_cos
0                  0  0.000000  1.000000
1                  1  0.258819  0.965926
2                  2  0.500000  0.866025


In [24]:
# ── Cell 6 — Lag Features ─────────────────────────────────────────────
grp = ts_df.groupby(['product_id','pincode'])['demand']

ts_df['lag_1']   = grp.shift(1)
ts_df['lag_24']  = grp.shift(24)
ts_df['lag_48']  = grp.shift(48)
ts_df['lag_168'] = grp.shift(168)

print("Lag features added!")
print(ts_df[['datetime','demand','lag_1','lag_24','lag_168']].head(10))

Lag features added!
             datetime  demand   lag_1  lag_24  lag_168
0 2024-01-01 00:00:00    78.0     NaN     NaN      NaN
1 2024-01-01 01:00:00    93.0    78.0     NaN      NaN
2 2024-01-01 02:00:00    39.0    93.0     NaN      NaN
3 2024-01-01 04:00:00    39.0    39.0     NaN      NaN
4 2024-01-01 05:00:00    31.0    39.0     NaN      NaN
5 2024-01-01 06:00:00   101.0    31.0     NaN      NaN
6 2024-01-01 07:00:00   451.0   101.0     NaN      NaN
7 2024-01-01 08:00:00   870.0   451.0     NaN      NaN
8 2024-01-01 09:00:00  1297.0   870.0     NaN      NaN
9 2024-01-01 10:00:00  1702.0  1297.0     NaN      NaN


In [25]:
# ── Cell 7 — Rolling Features ─────────────────────────────────────────
grp = ts_df.groupby(['product_id','pincode'])['demand']

ts_df['rolling_mean_24']  = grp.transform(
    lambda x: x.shift(1).rolling(24,  min_periods=1).mean())
ts_df['rolling_mean_168'] = grp.transform(
    lambda x: x.shift(1).rolling(168, min_periods=1).mean())
ts_df['rolling_mean_720'] = grp.transform(
    lambda x: x.shift(1).rolling(720, min_periods=1).mean())
ts_df['rolling_std_24']   = grp.transform(
    lambda x: x.shift(1).rolling(24,  min_periods=2).std().fillna(0))
ts_df['rolling_std_168']  = grp.transform(
    lambda x: x.shift(1).rolling(168, min_periods=2).std().fillna(0))
ts_df['rolling_max_24']   = grp.transform(
    lambda x: x.shift(1).rolling(24,  min_periods=1).max())
ts_df['rolling_max_168']  = grp.transform(
    lambda x: x.shift(1).rolling(168, min_periods=1).max())
ts_df['ewma_24']          = grp.transform(
    lambda x: x.shift(1).ewm(span=24,  adjust=False).mean())
ts_df['ewma_168']         = grp.transform(
    lambda x: x.shift(1).ewm(span=168, adjust=False).mean())

print("Rolling features added!")

Rolling features added!


In [26]:
# ── Cell 8 — Verify Features ──────────────────────────────────────────
print("=== FEATURE SUMMARY ===")
print(f"Total rows    : {len(ts_df):,}")
print(f"Total columns : {ts_df.shape[1]}")
print(f"\nColumns:")
for col in ts_df.columns.tolist():
    print(f"  → {col}")

print(f"\nMissing values:")
print(ts_df.isnull().sum()[ts_df.isnull().sum() > 0])

=== FEATURE SUMMARY ===
Total rows    : 2,059,538
Total columns : 32

Columns:
  → datetime
  → week_number
  → product_id
  → product_name
  → pincode
  → area_name
  → latitude
  → longitude
  → order_dow
  → order_hour_of_day
  → demand
  → demand_log
  → lag_168
  → hour_sin
  → hour_cos
  → dow_sin
  → dow_cos
  → week_sin
  → week_cos
  → month
  → lag_1
  → lag_24
  → lag_48
  → rolling_mean_24
  → rolling_mean_168
  → rolling_mean_720
  → rolling_std_24
  → rolling_std_168
  → rolling_max_24
  → rolling_max_168
  → ewma_24
  → ewma_168

Missing values:
lag_168             84000
lag_1                 500
lag_24              12000
lag_48              24000
rolling_mean_24       500
rolling_mean_168      500
rolling_mean_720      500
rolling_max_24        500
rolling_max_168       500
ewma_24               500
ewma_168              500
dtype: int64


In [27]:
# ── Cell 9 — Drop NaN Rows ────────────────────────────────────────────
FEATURES = [
    'hour_sin', 'hour_cos',
    'dow_sin',  'dow_cos',
    'week_sin', 'week_cos',
    'month',
    'lag_1', 'lag_24', 'lag_48', 'lag_168',
    'rolling_mean_24', 'rolling_mean_168', 'rolling_mean_720',
    'rolling_std_24',  'rolling_std_168',
    'rolling_max_24',  'rolling_max_168',
    'ewma_24',         'ewma_168',
]

before = len(ts_df)
ts_df  = ts_df.dropna(subset=FEATURES).copy()
after  = len(ts_df)

print(f"Rows before dropna : {before:,}")
print(f"Rows after dropna  : {after:,}")
print(f"Rows dropped       : {before-after:,}")

Rows before dropna : 2,059,538
Rows after dropna  : 1,975,538
Rows dropped       : 84,000


In [28]:
# ── Cell 10 — Fetch Weather ───────────────────────────────────────────
import requests

def fetch_weather(lat, lon, start='2024-01-01', end='2024-06-30'):
    url    = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude"  : lat,
        "longitude" : lon,
        "start_date": start,
        "end_date"  : end,
        "hourly"    : "temperature_2m,precipitation,relative_humidity_2m",
        "timezone"  : "Asia/Kolkata"
    }
    response = requests.get(url, params=params)
    data     = response.json()
    
    return pd.DataFrame({
        'datetime'   : pd.to_datetime(data['hourly']['time']),
        'temperature': data['hourly']['temperature_2m'],
        'rainfall'   : data['hourly']['precipitation'],
        'humidity'   : data['hourly']['relative_humidity_2m'],
    })

print("Fetching Ahmedabad weather 2024...")
weather_df = fetch_weather(23.0225, 72.5714)

# ── Weather Categories ────────────────────────────────────────────────
weather_df['is_raining'] = (weather_df['rainfall']    >  0.5).astype(int)
weather_df['is_hot']     = (weather_df['temperature'] > 35.0).astype(int)
weather_df['is_cold']    = (weather_df['temperature'] < 23.0).astype(int)
weather_df['is_humid']   = (weather_df['humidity']    > 80.0).astype(int)

print(f"Weather shape : {weather_df.shape}")
print(f"Date range    : {weather_df['datetime'].min()} → {weather_df['datetime'].max()}")
print(weather_df.head(5))

weather_df.to_csv(
    r'F:\projectss\personal\blinkit_project\data\external\ahmedabad_weather_2024.csv',
    index=False
)
print("✅ Weather saved!")

Fetching Ahmedabad weather 2024...
Weather shape : (4368, 8)
Date range    : 2024-01-01 00:00:00 → 2024-06-30 23:00:00
             datetime  temperature  rainfall  humidity  is_raining  is_hot  \
0 2024-01-01 00:00:00         17.5       0.0        80           0       0   
1 2024-01-01 01:00:00         16.8       0.0        84           0       0   
2 2024-01-01 02:00:00         16.0       0.0        88           0       0   
3 2024-01-01 03:00:00         15.6       0.0        88           0       0   
4 2024-01-01 04:00:00         15.5       0.0        88           0       0   

   is_cold  is_humid  
0        1         0  
1        1         1  
2        1         1  
3        1         1  
4        1         1  
✅ Weather saved!


In [29]:
weather_df['is_cold'].value_counts()

is_cold
0    3352
1    1016
Name: count, dtype: int64

In [30]:
# ── Cell 11 — Merge Weather ───────────────────────────────────────────
weather_df['datetime'] = pd.to_datetime(weather_df['datetime'])
ts_df['datetime']      = pd.to_datetime(ts_df['datetime'])

# round to nearest hour for merging
ts_df['datetime_hour'] = ts_df['datetime'].dt.floor('h')
weather_df['datetime_hour'] = weather_df['datetime'].dt.floor('h')

ts_df = ts_df.merge(
    weather_df[['datetime_hour','temperature','rainfall',
                'humidity','is_raining','is_hot','is_cold','is_humid']],
    on  ='datetime_hour',
    how ='left'
)

ts_df = ts_df.drop(columns=['datetime_hour'])

print(f"Shape after weather merge : {ts_df.shape}")
print(f"Missing weather values    :")
print(ts_df[['temperature','rainfall','humidity']].isnull().sum())
print(ts_df[['datetime','temperature','rainfall','humidity','is_raining']].head(5))

Shape after weather merge : (1975538, 39)
Missing weather values    :
temperature    0
rainfall       0
humidity       0
dtype: int64
             datetime  temperature  rainfall  humidity  is_raining
0 2024-01-08 01:00:00         18.5       0.0        78           0
1 2024-01-08 02:00:00         18.0       0.0        82           0
2 2024-01-08 04:00:00         17.3       0.0        87           0
3 2024-01-08 05:00:00         17.0       0.0        88           0
4 2024-01-08 06:00:00         18.0       0.0        84           0


In [31]:
# ── Cell 12 — Festival Features ──────────────────────────────────────
festivals_2024 = {
    # Indian festivals
    '2024-01-14': 'Makar Sankranti',
    '2024-01-22': 'Ram Mandir',
    '2024-01-26': 'Republic Day',
    '2024-03-25': 'Holi',
    '2024-04-14': 'Ambedkar Jayanti',
    '2024-04-17': 'Ram Navami',
    '2024-05-23': 'Buddha Purnima',
    '2024-06-17': 'Eid ul Adha',

    # Gujarat specific
    '2024-01-15': 'Uttarayan',       # kite festival — huge in Ahmedabad!
    '2024-10-02': 'Gandhi Jayanti',
    '2024-10-12': 'Navratri Start',  # very big in Gujarat!
    '2024-10-13': 'Navratri',
    '2024-10-14': 'Navratri',
    '2024-10-15': 'Navratri',
    '2024-10-16': 'Navratri',
    '2024-10-17': 'Navratri',
    '2024-10-18': 'Navratri',
    '2024-10-19': 'Navratri',
    '2024-10-20': 'Navratri',
    '2024-10-21': 'Navratri',
    '2024-11-01': 'Diwali',
    '2024-11-02': 'Diwali',
    '2024-11-03': 'Diwali',
    '2024-12-25': 'Christmas',
    '2024-12-31': 'New Year Eve',
}

festival_df = pd.DataFrame([
    {'date': pd.Timestamp(date), 'festival_name': name}
    for date, name in festivals_2024.items()
])

festival_df['is_festival'] = 1

# add date column to ts_df for merging
ts_df['date'] = ts_df['datetime'].dt.date.astype('datetime64[ns]')

ts_df = ts_df.merge(
    festival_df,
    left_on  ='date',
    right_on ='date',
    how      ='left'
)

ts_df['is_festival']   = ts_df['is_festival'].fillna(0).astype(int)
ts_df['festival_name'] = ts_df['festival_name'].fillna('none')

print(f"Shape after festival merge : {ts_df.shape}")
print(f"Festival rows              : {ts_df['is_festival'].sum():,}")
print(ts_df[ts_df['is_festival']==1][['datetime','festival_name']].head(10))

Shape after festival merge : (1975538, 42)
Festival rows              : 102,091
               datetime    festival_name
142 2024-01-14 00:00:00  Makar Sankranti
143 2024-01-14 01:00:00  Makar Sankranti
144 2024-01-14 02:00:00  Makar Sankranti
145 2024-01-14 03:00:00  Makar Sankranti
146 2024-01-14 04:00:00  Makar Sankranti
147 2024-01-14 05:00:00  Makar Sankranti
148 2024-01-14 06:00:00  Makar Sankranti
149 2024-01-14 07:00:00  Makar Sankranti
150 2024-01-14 08:00:00  Makar Sankranti
151 2024-01-14 09:00:00  Makar Sankranti


In [32]:
# ── Cell 13 — Extra Time Features ────────────────────────────────────
ts_df['is_weekend']    = (ts_df['order_dow'] >= 5).astype(int)
ts_df['is_peak_hour']  = (ts_df['order_hour_of_day'].between(11, 14) |
                          ts_df['order_hour_of_day'].between(18, 21)).astype(int)
ts_df['is_night']      = (ts_df['order_hour_of_day'].between(22, 23) |
                          ts_df['order_hour_of_day'].between(0,  5)).astype(int)
ts_df['is_morning']    = (ts_df['order_hour_of_day'].between(6, 10)).astype(int)

print("Extra time features added!")
print(ts_df[['order_hour_of_day','is_weekend',
             'is_peak_hour','is_night','is_morning']].head(10))

Extra time features added!
   order_hour_of_day  is_weekend  is_peak_hour  is_night  is_morning
0                  1           0             0         1           0
1                  2           0             0         1           0
2                  4           0             0         1           0
3                  5           0             0         1           0
4                  6           0             0         0           1
5                  7           0             0         0           1
6                  8           0             0         0           1
7                  9           0             0         0           1
8                 10           0             0         0           1
9                 11           0             1         0           0


In [33]:
# ── Cell 14 — Update Features List ───────────────────────────────────
FEATURES = [
    # time features
    'hour_sin', 'hour_cos',
    'dow_sin',  'dow_cos',
    'week_sin', 'week_cos',
    'month',

    # lag features
    'lag_1', 'lag_24', 'lag_48', 'lag_168',

    # rolling features
    'rolling_mean_24', 'rolling_mean_168', 'rolling_mean_720',
    'rolling_std_24',  'rolling_std_168',
    'rolling_max_24',  'rolling_max_168',
    'ewma_24',         'ewma_168',

    # weather features  ← NEW
    'temperature', 'rainfall', 'humidity',
    'is_raining',  'is_hot',   'is_cold', 'is_humid',

    # festival features ← NEW
    'is_festival', 'is_weekend',

    # time of day features ← NEW
    'is_peak_hour', 'is_night', 'is_morning',
]

print(f"Total features : {len(FEATURES)}")
print(f"Features list  : {FEATURES}")

Total features : 32
Features list  : ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'week_sin', 'week_cos', 'month', 'lag_1', 'lag_24', 'lag_48', 'lag_168', 'rolling_mean_24', 'rolling_mean_168', 'rolling_mean_720', 'rolling_std_24', 'rolling_std_168', 'rolling_max_24', 'rolling_max_168', 'ewma_24', 'ewma_168', 'temperature', 'rainfall', 'humidity', 'is_raining', 'is_hot', 'is_cold', 'is_humid', 'is_festival', 'is_weekend', 'is_peak_hour', 'is_night', 'is_morning']


In [34]:
# ── Cell 10 — Save ────────────────────────────────────────────────────
ts_df.to_csv(
    r'F:\projectss\personal\blinkit_project\data\processed\ts_features.csv',
    index=False
)

print(f"✅ Saved!")
print(f"Shape   : {ts_df.shape}")
print(f"Columns : {ts_df.columns.tolist()}")

✅ Saved!
Shape   : (1975538, 46)
Columns : ['datetime', 'week_number', 'product_id', 'product_name', 'pincode', 'area_name', 'latitude', 'longitude', 'order_dow', 'order_hour_of_day', 'demand', 'demand_log', 'lag_168', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'week_sin', 'week_cos', 'month', 'lag_1', 'lag_24', 'lag_48', 'rolling_mean_24', 'rolling_mean_168', 'rolling_mean_720', 'rolling_std_24', 'rolling_std_168', 'rolling_max_24', 'rolling_max_168', 'ewma_24', 'ewma_168', 'temperature', 'rainfall', 'humidity', 'is_raining', 'is_hot', 'is_cold', 'is_humid', 'date', 'festival_name', 'is_festival', 'is_weekend', 'is_peak_hour', 'is_night', 'is_morning']
